# Notebook 00 - Setup and Storage Overview

## Purpose

This notebook introduces the setup and storage model for using `fintech-market-ingestion` and future `stratlake-trade-engine` workflows in a temporary notebook environment such as Google Colab.

It orients the tutorial sequence around the current milestone deployment pattern: work under the local Colab runtime, persist selectively to Google Drive, and use archive backup packs for larger curated market datasets.

This notebook is a setup, storage, validation, and review layer. It does not perform market-data ingestion, StratLake feature generation, archive/restore execution, strategy execution, backtesting, or artifact generation.

## Upstream apps used

- `fintech-market-ingestion`: used for setup, native CLI sanity checks, workspace initialization, session-save previews, and archive backup previews.
- `stratlake-trade-engine`: referenced as a later downstream workflow consumer; no StratLake feature, strategy, backtest, archive, or artifact behavior is implemented here.

## Workflow role

- setup
- session persistence
- archive
- restore
- validation
- audit/review

## Runtime assumptions

- Designed for Google Colab.
- Active runtime work happens under `/content`.
- Google Drive is used only for persistence, backup, archive, and restore workflows.
- Generated data, archives, restore packs, local workspaces, runtime folders, notebook outputs, and secrets are not committed.

## Required user inputs

- `<DRIVE_FOLDER_PLACEHOLDER>`: Google Drive folder for tutorial persistence, backup, archive, and restore storage.
- `<FINTECH_SESSION_NAME>`: Fintech runtime session name used by later notebooks.
- `<ARCHIVE_ID>`: Archive identifier used by later archive/restore notebooks.
- `<ALPACA_API_KEY_ID>` and `<ALPACA_API_SECRET_KEY>`: Colab Secret names for later extraction notebooks, not literal credential values.

## Secrets

This notebook must use Colab Secrets or safe runtime prompts for credentials.

Do not hard-code API keys, tokens, secrets, `.env` values, credential JSON, or private keys.

## Path conventions

Use portable runtime variables. Do not hard-code local machine paths, personal Google Drive paths, usernames, or committed local workspace paths.

Recommended variables:

- `CONTENT_ROOT`
- `LOCAL_WORKSPACE`
- `LOCAL_CURATED`
- `DRIVE_PROJECT_ROOT`
- `DRIVE_SESSION_ROOT`
- `DRIVE_BACKUP_ROOT`
- `FINTECH_SESSION_NAME`
- `ARCHIVE_ID`

## Native-command-first boundary

This notebook should orchestrate setup, call native upstream CLI help or dry-run commands, document path conventions, validate runtime assumptions, and guide human review.

It must not reimplement native ingestion, archive/restore, feature generation, strategy, backtest, or artifact logic. Later notebooks should call upstream app CLI commands where available.

## Generated artifact boundaries

This notebook may create runtime directories when executed, but generated files must stay outside Git.

- Fintech runtime workspace: under `/content`, not committed.
- Google Drive persistence folder: used for backup, archive, and restore storage, not Git source.
- Notebook outputs: cleared before commit.

Do not commit generated data, archives, restore packs, local app workspaces, runtime folders, notebook outputs, or secrets.

## Validation before commit

Run:

```bash
python scripts/scan_for_secret_patterns.py .
python scripts/check_notebooks_no_outputs.py notebooks
python scripts/validate_repo_cleanliness.py .
```


## Tutorial Series Map

This notebook is the orientation step for the full tutorial series.

| Notebook | Theme | Purpose |
|---|---|---|
| `00_setup_and_storage_overview.ipynb` | Setup and storage model | Prepare the workspace and explain latest deployment storage choices |
| `01_extraction_daily_bars_backfill.ipynb` | Extraction layer | Pull daily bars from Alpaca into local curated storage |
| `02_session_save_and_restore.ipynb` | Session persistence | Save and restore lightweight project state |
| `03_archive_backup_pack_and_restore.ipynb` | Archive persistence | Store, validate, and restore larger curated datasets efficiently |
| `04_storage_strategy_comparison.ipynb` | Decision guide | Choose the right persistence pattern for a given situation |

This notebook should stay focused on setup, storage boundaries, and deployment readiness. The extraction, session restore, and archive restore workflows are easier to teach in separate notebooks.

## Storage Model

The tutorial uses three storage areas:

| Storage area | Example path | Purpose |
|---|---|---|
| Local workspace | `/content/fintech-market-ingestion-demo` | Active notebook workspace where commands run |
| Local curated data | `/content/fintech-market-ingestion-demo/data/curated` | Working market data used by extraction and analysis |
| Google Drive storage | `/content/drive/MyDrive/<DRIVE_FOLDER_PLACEHOLDER>` | Persistence layer for saved sessions and archive packs |

The local workspace is the active working layer. Google Drive is a persistence layer used to save or restore selected outputs between notebook sessions.

## Latest milestone deployment emphasis

The latest storage workflow should avoid repeatedly copying thousands of small partitioned Parquet files between Colab and Drive. Instead:

1. Run extraction and analysis against the local workspace.
2. Save lightweight project/session state when you want continuity.
3. Pack larger curated datasets into archive backup packs when you want faster persistence and restore.
4. Restore archive packs back into local workspace storage before running downstream notebooks.

## Install Package Dependencies

Install the package and any required supporting dependencies for this notebook environment.

The package installation command below uses TestPyPI because this tutorial is designed around the current project release workflow. If you publish the package to standard PyPI later, replace the install command with the normal PyPI package installation.

After installation, the next cell verifies that the deployment exposes the expected notebook/storage CLIs.

In [ ]:
# Install package dependencies in the active Colab runtime.
# This uses native package installation only; it does not run ingestion or generate data.
!python -m pip install --upgrade pip
!python -m pip install "pandas-market-calendars>=5.0"
!python -m pip install -i https://test.pypi.org/simple/ fintech-market-ingestion


## Verify Latest Deployment CLIs

Before creating folders or saving data, confirm the installed deployment exposes the expected native commands.

This notebook uses the following CLI families:

- `fintech-init-project` for local workspace initialization
- `fintech-save-session` for lightweight session persistence
- `fintech-backup-data` for archive pack creation, validation, and restore planning

The help output is useful during runtime because it makes the deployed command surface visible before later cells rely on it. Clear outputs before committing the notebook.


In [ ]:
!fintech-init-project --help
!fintech-save-session --help
!fintech-backup-data --help


## Initialize the Local Project Workspace

Create a local project workspace for the notebook demonstration.

This workspace is where tutorial commands should run. It is the active local environment for configs, curated data, reports, artifacts, and session metadata.

The workflow should keep execution local even when Google Drive is mounted. Drive paths are used for persistence targets only.


In [ ]:
FINTECH_SESSION_NAME = "<FINTECH_SESSION_NAME>"

!fintech-init-project   --root /content/fintech-market-ingestion-demo   --notebooks   --with-session   --session-name "{FINTECH_SESSION_NAME}"


## Define Notebook Paths

Use shared path constants so later cells are easy to adapt.

The local workspace remains the active execution root. The Drive project root is the persistence root for saved sessions and archive backup packs.


In [ ]:
from pathlib import Path

CONTENT_ROOT = Path("/content")
LOCAL_WORKSPACE = CONTENT_ROOT / "fintech-market-ingestion-demo"
LOCAL_CURATED = LOCAL_WORKSPACE / "data" / "curated"

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/<DRIVE_FOLDER_PLACEHOLDER>")
DRIVE_SESSION_ROOT = DRIVE_PROJECT_ROOT / "sessions" / FINTECH_SESSION_NAME
DRIVE_BACKUP_ROOT = DRIVE_PROJECT_ROOT / "backups"

ARCHIVE_ID = "<ARCHIVE_ID>"

for runtime_path in (CONTENT_ROOT, LOCAL_WORKSPACE, LOCAL_CURATED):
    if not runtime_path.as_posix().startswith("/content"):
        raise ValueError(f"Active runtime path must stay under /content: {runtime_path}")


## Configure Alpaca Secret Names

Later notebooks that extract market data from Alpaca need API credentials.

In Colab, the safest pattern is to store sensitive values in Colab Secrets rather than hardcoding them in notebook cells. This setup notebook records the expected secret names only; it does not require or print credential values.


In [ ]:
import getpass
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_secret_or_prompt(name: str) -> str:
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass.getpass(f"Enter {name}: ")
    return value


ALPACA_API_KEY_ID_SECRET_NAME = "ALPACA_API_KEY_ID"
ALPACA_API_SECRET_KEY_SECRET_NAME = "ALPACA_API_SECRET_KEY"
ALPACA_DATA_BASE_URL = "https://data.alpaca.markets"
ALPACA_FEED = "iex"


## Mount Google Drive for Persistent Storage

For this notebook demonstration, mount Google Drive so selected session outputs and archive packs can persist beyond the current Colab runtime.

Without a persistent storage layer, a reset runtime may require rerunning extraction commands to recreate backfilled data. Persisting selected data and outputs makes it easier to resume later.

Google Drive is used here as a convenience storage location only. The active workflow should still run from the local notebook workspace.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## Verify Google Drive Access

After mounting Drive, confirm that the expected MyDrive root is available. Do not print directory listings or personal Drive file names in committed notebook outputs.


In [ ]:
drive_root = Path("/content/drive/MyDrive")
if not drive_root.exists():
    raise RuntimeError("Google Drive root is not available. Mount Drive before continuing.")


## Create Persistent Storage Directories

Create the basic Google Drive directories used by later notebooks.

The workflow separates lightweight session persistence from larger archive backup packs:

- `sessions/` stores project/session continuity outputs.
- `backups/` stores archive packs for curated datasets.

The command uses `exist_ok=True`, so rerunning it is safe. These folders are runtime and Drive persistence locations, not Git source.


In [ ]:
DRIVE_SESSION_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BACKUP_ROOT.mkdir(parents=True, exist_ok=True)


## Session Saving vs Archive Backup Packs

`fintech-market-ingestion` supports two related but different persistence patterns:

1. **Session saving** — preserve selected project/session files so a notebook or local workflow can be resumed.
2. **Archive backup packs** — package larger curated datasets into transfer-friendly backup packs for faster restore.

Both patterns treat Google Drive as a mounted filesystem path only. Neither pattern makes Google Drive, copied files, archive packs, or manifests the canonical source of truth.

| Storage pattern | Best for | Main benefit | Typical source | Typical destination |
|---|---|---|---|---|
| Session saving | Project/session continuity | Easy resume of configs, reports, artifacts, notebooks, and selected outputs | Local workspace | Drive `sessions/` |
| Archive backup packs | Large curated data snapshots | Faster transfer and restore of many partitioned files | Local `data/curated/` | Drive `backups/` |

## Rule of thumb

> Session save = preserve workflow state.  
> Archive backup pack = preserve larger backfilled datasets efficiently.  
> Restore archive packs locally before analysis rather than running against Drive-hosted shards.

## Session Save Preview

Session saving is best for lightweight project state such as reports, artifacts, configs, notebooks, and session metadata.

The command below is shown as a **dry-run preview**. It plans the save operation without copying files.

In [ ]:
!fintech-save-session   --root "{LOCAL_WORKSPACE}"   --session-id "{FINTECH_SESSION_NAME}"   --policy artifacts_and_reports   --adapter google-drive   --destination "{DRIVE_SESSION_ROOT}"   --dry-run


## Archive Backup Pack Preview

Archive backup packs are best for larger curated datasets, especially partitioned Parquet data.

Use this pattern after a backfill or data preparation notebook has created local curated data under `data/curated/`.

The command below is also a dry-run preview. It plans the backup pack without writing archive shards.


In [ ]:
!fintech-backup-data pack   --workspace-root "{LOCAL_WORKSPACE}"   --source-dataset-root "{LOCAL_CURATED}"   --backup-root "{DRIVE_BACKUP_ROOT}"   --backup-id "{ARCHIVE_ID}"   --shard-size-mb 512   --dry-run


## Archive Validate and Restore Readiness

After a real archive pack is created in a later notebook, validate the pack before relying on it for restore.

This setup notebook shows the deployed validation command surface. Run a real validation in the archive notebook after a pack exists.


In [ ]:
!fintech-backup-data validate --help


## Archive Restore Preview

The restore pattern should copy archived curated data back into the local workspace before downstream notebooks run.

This preserves the storage principle: Drive is the persistence layer; local notebook storage is the active execution layer.

The cell below prints the restore command template instead of running it. Run the real restore command in the archive notebook after a pack exists.


In [ ]:
restore_command = (
    "fintech-backup-data restore "
    f"--workspace-root {LOCAL_WORKSPACE} "
    f"--target-dataset-root {LOCAL_CURATED} "
    f"--backup-root {DRIVE_BACKUP_ROOT} "
    f"--backup-id {ARCHIVE_ID}"
)

print(restore_command)


## What Comes Next

After this setup notebook, continue with the extraction tutorial.

The next notebook should introduce the daily bars backfill command and write market data into the local curated dataset:

```text
/content/fintech-market-ingestion-demo/data/curated
```

Once data exists locally, later notebooks can demonstrate:

- saving session state to Google Drive
- restoring saved session state
- creating archive backup packs from local curated data
- validating archive backup packs
- restoring archive backup packs back into local curated storage
- preparing curated data for StratLake workflows
- choosing the right persistence method for a given situation

## Summary

This notebook prepared the local workspace and explained the storage model.

The key idea is:

> The local workspace is where work happens.  
> Google Drive is where selected session state and backup packs persist.  
> Session saving preserves project state.  
> Archive backup packs preserve larger datasets efficiently.  
> Restore archives locally before analysis.
